# 02 · Generación de datos sintéticos y carga en BigQuery

Genera datos de negocio coherentes con `Faker` para las 7 tablas y las carga en BigQuery
respetando el orden de dependencias FK. 


In [2]:
import os
import random
import datetime
import unicodedata
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account

load_dotenv()
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

if CREDENTIALS_PATH:
    credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
    client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
else:
    client = bigquery.Client(project=PROJECT_ID)

random.seed(42)
np.random.seed(42)
Faker.seed(42)
print(f"Conectado a {client.project}.{DATASET_ID}")

Conectado a sqlproyect-507620.0


## Países, locales y ciudades

Usamos varios locales de `Faker` para que nombres, teléfonos y direcciones sean coherentes con el país de cada cliente.

In [3]:
COUNTRIES = {
    "España":        {"locale": "es_ES", "cities": ["Madrid", "Barcelona", "Valencia", "Sevilla", "Bilbao", "Málaga"]},
    "Francia":       {"locale": "fr_FR", "cities": ["París", "Lyon", "Marsella", "Toulouse", "Niza"]},
    "Alemania":      {"locale": "de_DE", "cities": ["Berlín", "Múnich", "Hamburgo", "Colonia", "Fráncfort"]},
    "Italia":        {"locale": "it_IT", "cities": ["Roma", "Milán", "Nápoles", "Turín", "Bolonia"]},
    "Portugal":      {"locale": "pt_PT", "cities": ["Lisboa", "Oporto", "Braga", "Coimbra"]},
    "Países Bajos":  {"locale": "nl_NL", "cities": ["Ámsterdam", "Róterdam", "La Haya", "Utrecht"]},
    "Bélgica":       {"locale": "nl_BE", "cities": ["Bruselas", "Amberes", "Gante"]},
    "Polonia":       {"locale": "pl_PL", "cities": ["Varsovia", "Cracovia", "Wrocław", "Poznań"]},
}
# España, Alemania y Francia pesan más -> mercados principales
COUNTRY_WEIGHTS = [0.28, 0.16, 0.18, 0.12, 0.08, 0.08, 0.05, 0.05]

fakers_by_locale = {info["locale"]: Faker(info["locale"]) for info in COUNTRIES.values()}
fake_generic = Faker()

START_HISTORY = datetime.datetime(2023, 1, 1)
END_HISTORY = datetime.datetime(2026, 8, 31)

def random_dt_between(start, end):
    delta = end - start
    return start + datetime.timedelta(seconds=random.randint(0, int(delta.total_seconds())))

## 1. `customers` (500)

In [ ]:
N_CUSTOMERS = 500
ACQUISITION_CHANNELS = ["organic", "paid_ads", "social_media", "referral", "email_marketing", "affiliate"]
ACQ_WEIGHTS = [0.30, 0.25, 0.20, 0.12, 0.08, 0.05]

def _clean(s):
    # quita tildes/diacríticos para que el email sea ASCII
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s.lower().replace(" ", "").replace("'", "")

customers_rows = []
for i in range(1, N_CUSTOMERS + 1):
    country = random.choices(list(COUNTRIES.keys()), weights=COUNTRY_WEIGHTS, k=1)[0]
    info = COUNTRIES[country]
    fk = fakers_by_locale[info["locale"]]
    first_name, last_name = fk.first_name(), fk.last_name()
    reg_dt = random_dt_between(START_HISTORY, END_HISTORY)
    customers_rows.append({
        "customer_id": i,
        "first_name": first_name,
        "last_name": last_name,
        "email": f"{_clean(first_name)}.{_clean(last_name)}{i}@{fake_generic.free_email_domain()}",
        "phone": fk.phone_number(),
        "country": country,
        "city": random.choice(info["cities"]),
        "acquisition_channel": random.choices(ACQUISITION_CHANNELS, weights=ACQ_WEIGHTS, k=1)[0],
        "registration_date": reg_dt.date(),
        "created_at": reg_dt,
    })

customers_df = pd.DataFrame(customers_rows)
assert customers_df["customer_id"].is_unique and customers_df["email"].is_unique
customers_df.head()

,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date,created_at
0,1,Eugenia,Caracciolo,eugenia.caracciolo1@gmail.com,0481321810,Italia,Nápoles,organic,2023-02-08,2023-02-08 20:28:06
1,2,Ciro,Gil,ciro.gil2@gmail.com,+34 982908386,España,Málaga,social_media,2023-06-09,2023-06-09 05:17:49
2,3,Heinz-Werner,Meister,heinz-werner.meister3@hotmail.com,+49(0)0265 42351,Alemania,Colonia,organic,2025-07-06,2025-07-06 07:09:23
3,4,Cirino,Pinto,cirino.pinto4@gmail.com,+34 927594078,España,Bilbao,social_media,2023-12-28,2023-12-28 10:13:36
4,5,Faruk,Kostolzin,faruk.kostolzin5@gmail.com,04959 31034,Alemania,Fráncfort,paid_ads,2026-01-17,2026-01-17 07:08:45


## 2. `categories` (8, lista fija de negocio)

In [5]:
categories_list = [
    ("Smartphones", "Teléfonos móviles y accesorios de telefonía"),
    ("Laptops", "Portátiles para uso personal, gaming y profesional"),
    ("Audio", "Auriculares, altavoces y equipos de sonido"),
    ("Wearables", "Smartwatches y pulseras de actividad"),
    ("Tablets", "Tablets y accesorios"),
    ("Gaming", "Consolas, periféricos y accesorios gaming"),
    ("Fotografía", "Cámaras y accesorios fotográficos"),
    ("Accesorios", "Cables, fundas, cargadores y otros accesorios"),
]
categories_df = pd.DataFrame([
    {"category_id": i + 1, "category_name": name, "description": desc}
    for i, (name, desc) in enumerate(categories_list)
])
categories_df

,category_id,category_name,description
0,1,Smartphones,Teléfonos móviles y accesorios de telefonía
1,2,Laptops,"Portátiles para uso personal, gaming y profesi..."
2,3,Audio,"Auriculares, altavoces y equipos de sonido"
3,4,Wearables,Smartwatches y pulseras de actividad
4,5,Tablets,Tablets y accesorios
5,6,Gaming,"Consolas, periféricos y accesorios gaming"
6,7,Fotografía,Cámaras y accesorios fotográficos
7,8,Accesorios,"Cables, fundas, cargadores y otros accesorios"


## 3. `products` (70)

Repartidos entre categorías, con rangos de precio realistas por categoría y `cost < sale_price`.

In [6]:
PRODUCT_TEMPLATES = {
    "Smartphones": {"brands": ["Novaphone", "Zentek", "Orbitel", "Lumea"], "models": ["X1", "X2 Pro", "Nova 5", "Edge 12", "Air S"], "price": (199, 1299)},
    "Laptops": {"brands": ["Compunova", "Vertex", "AeroTech", "Bytewave"], "models": ["Book 14", "Pro 16", "Slim S3", "Gamer G7"], "price": (399, 2499)},
    "Audio": {"brands": ["SoundWave", "EchoLabs", "BassPoint"], "models": ["Buds Air", "Studio Max", "Pulse 2", "Boom Mini"], "price": (19, 349)},
    "Wearables": {"brands": ["FitTrack", "PulseWear", "Chronos"], "models": ["Watch S", "Band 6", "Active Pro"], "price": (29, 449)},
    "Tablets": {"brands": ["Tabula", "Compunova", "AeroTech"], "models": ["Pad 10", "Pad Pro 12", "Mini 8"], "price": (149, 999)},
    "Gaming": {"brands": ["Playtron", "GameForge", "NeoPlay"], "models": ["Controller X", "Headset Pro", "Console Z"], "price": (24, 599)},
    "Fotografía": {"brands": ["LensCraft", "PixelPro", "OptiCam"], "models": ["Mirrorless M1", "Compact C2", "Action Cam"], "price": (89, 1799)},
    "Accesorios": {"brands": ["ChargeIt", "CablePro", "CasePlus"], "models": ["Cable USB-C", "Cargador 65W", "Funda Slim"], "price": (5, 79)},
}

N_PRODUCTS = 70
cat_names = list(PRODUCT_TEMPLATES.keys())
base_per_cat = N_PRODUCTS // len(cat_names)
remainder = N_PRODUCTS - base_per_cat * len(cat_names)
counts = {c: base_per_cat for c in cat_names}
for c in random.sample(cat_names, remainder):
    counts[c] += 1

products_rows, pid = [], 1
for cat_name, n in counts.items():
    cat_id = int(categories_df.loc[categories_df.category_name == cat_name, "category_id"].iloc[0])
    tmpl = PRODUCT_TEMPLATES[cat_name]
    used_names = set()
    for _ in range(n):
        name = f"{random.choice(tmpl['brands'])} {random.choice(tmpl['models'])}"
        base_name, suffix = name, 1
        while name in used_names:
            suffix += 1
            name = f"{base_name} v{suffix}"
        used_names.add(name)
        price = round(random.uniform(*tmpl["price"]), 2)
        cost = round(price * random.uniform(0.45, 0.70), 2)
        products_rows.append({
            "product_id": pid,
            "category_id": cat_id,
            "product_name": name,
            "description": f"{name} - producto de la categoría {cat_name}",
            "sale_price": price,
            "cost": cost,
            "stock_quantity": random.randint(0, 500),
            "is_active": random.random() < 0.92,
            "created_at": random_dt_between(START_HISTORY, START_HISTORY + datetime.timedelta(days=200)),
        })
        pid += 1

products_df = pd.DataFrame(products_rows)
assert products_df["product_id"].is_unique
assert (products_df["cost"] < products_df["sale_price"]).all()
products_df.head()

,product_id,category_id,product_name,description,sale_price,cost,stock_quantity,is_active,created_at
0,1,1,Zentek Nova 5,Zentek Nova 5 - producto de la categoría Smart...,1223.51,648.01,201,False,2023-04-27 05:03:58
1,2,1,Orbitel Air S,Orbitel Air S - producto de la categoría Smart...,1110.12,647.29,362,True,2023-07-04 10:19:11
2,3,1,Novaphone Nova 5,Novaphone Nova 5 - producto de la categoría Sm...,563.36,268.95,213,True,2023-01-11 08:22:10
3,4,1,Lumea Nova 5,Lumea Nova 5 - producto de la categoría Smartp...,1262.78,815.41,397,True,2023-03-30 08:51:23
4,5,1,Novaphone Air S,Novaphone Air S - producto de la categoría Sma...,727.49,422.71,369,True,2023-05-28 15:37:50


## 4. `orders` (2000)

La fecha de pedido nunca es anterior al registro del cliente. Envío/entrega se derivan del `order_status`.

In [7]:
N_ORDERS = 2000
ORDER_STATUSES = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
STATUS_WEIGHTS = [0.07, 0.08, 0.10, 0.55, 0.10, 0.10]

cust_by_id = customers_df.set_index("customer_id")
orders_rows = []
for oid in range(1, N_ORDERS + 1):
    cust_id = random.randint(1, N_CUSTOMERS)
    cust = cust_by_id.loc[cust_id]
    reg_dt = cust["created_at"]
    order_dt = random_dt_between(max(reg_dt, START_HISTORY), END_HISTORY)
    status = random.choices(ORDER_STATUSES, weights=STATUS_WEIGHTS, k=1)[0]

    same_address = random.random() < 0.9
    ship_country = cust["country"] if same_address else random.choices(list(COUNTRIES.keys()), weights=COUNTRY_WEIGHTS, k=1)[0]
    ship_city = cust["city"] if ship_country == cust["country"] else random.choice(COUNTRIES[ship_country]["cities"])

    shipped_dt = delivered_dt = None
    if status in ("shipped", "delivered", "returned"):
        shipped_dt = order_dt + datetime.timedelta(days=random.randint(1, 3), hours=random.randint(0, 12))
    if status in ("delivered", "returned"):
        delivered_dt = shipped_dt + datetime.timedelta(days=random.randint(1, 5))

    orders_rows.append({
        "order_id": oid,
        "customer_id": int(cust_id),
        "order_status": status,
        "shipping_address": fakers_by_locale[COUNTRIES[ship_country]["locale"]].street_address(),
        "shipping_city": ship_city,
        "shipping_country": ship_country,
        "order_date": order_dt,
        "shipped_date": shipped_dt,
        "delivered_date": delivered_dt,
    })

orders_df = pd.DataFrame(orders_rows)
assert orders_df["order_id"].is_unique
assert orders_df["customer_id"].isin(customers_df["customer_id"]).all()
orders_df.head()

,order_id,customer_id,order_status,shipping_address,shipping_city,shipping_country,order_date,shipped_date,delivered_date
0,1,137,delivered,Acceso de Víctor Salvà 10 Piso 9,Valencia,España,2026-06-04 15:05:35,2026-06-06 22:05:35,2026-06-11 22:05:35
1,2,125,delivered,Cañada Adolfo Mas 63,Madrid,España,2026-03-01 21:33:49,2026-03-03 07:33:49,2026-03-04 07:33:49
2,3,48,delivered,Paseo de Ariadna Cortes 1,Valencia,España,2024-11-09 09:01:47,2024-11-10 19:01:47,2024-11-11 19:01:47
3,4,360,confirmed,"R. de Brito, 45",Lisboa,Portugal,2025-12-20 20:33:28,NaT,NaT
4,5,331,delivered,"Contrada Busoni, 9",Nápoles,Italia,2026-04-20 22:08:44,2026-04-22 06:08:44,2026-04-23 06:08:44


## 5. `order_items` (~4500, media 2-3 productos/pedido)

Cada producto aparece como mucho una vez por pedido (`UNIQUE(order_id, product_id)`). `unit_price` es el precio en el momento de la compra: se simula con una variación de ±5% sobre el precio actual del catálogo.

In [8]:
active_products = products_df[products_df.is_active].product_id.tolist()
all_products = products_df.product_id.tolist()
prod_price = products_df.set_index("product_id")["sale_price"]

order_items_rows, oiid = [], 1
for _, order in orders_df.iterrows():
    n_items = np.random.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])
    pool = active_products if len(active_products) >= n_items else all_products
    chosen = random.sample(pool, k=min(n_items, len(pool)))
    for prod_id in chosen:
        qty = random.choices([1, 2, 3], weights=[0.65, 0.25, 0.10], k=1)[0]
        base_price = float(prod_price.loc[prod_id])
        unit_price = round(base_price * random.uniform(0.95, 1.05), 2)
        discount = round(unit_price * qty * random.choice([0, 0, 0, 0.05, 0.10, 0.15]), 2)
        order_items_rows.append({
            "order_item_id": oiid,
            "order_id": int(order["order_id"]),
            "product_id": int(prod_id),
            "quantity": int(qty),
            "unit_price": unit_price,
            "discount_amount": discount,
        })
        oiid += 1

order_items_df = pd.DataFrame(order_items_rows)
assert order_items_df["order_item_id"].is_unique
assert not order_items_df.duplicated(subset=["order_id", "product_id"]).any()
print(f"order_items: {len(order_items_df)} filas | media {len(order_items_df)/N_ORDERS:.2f} por pedido")
order_items_df.head()

order_items: 4586 filas | media 2.29 por pedido


,order_item_id,order_id,product_id,quantity,unit_price,discount_amount
0,1,1,4,2,1244.34,248.87
1,2,1,30,2,447.10,89.42
2,3,2,8,1,738.78,0.00
3,4,2,52,1,601.32,0.00
4,5,2,34,2,118.16,0.00


## 6. `payments`

Un pago por pedido. El importe cuadra con la suma de sus líneas, y el estado del pago está correlado con el estado del pedido.

In [9]:
PAYMENT_METHODS = ["credit_card", "paypal", "bank_transfer", "other"]
PM_WEIGHTS = [0.45, 0.30, 0.15, 0.10]

order_totals = (
    order_items_df.assign(line_total=lambda d: d.quantity * d.unit_price - d.discount_amount)
    .groupby("order_id")["line_total"].sum()
)

payments_rows, pmid = [], 1
for _, order in orders_df.iterrows():
    oid, status = order["order_id"], order["order_status"]
    amount = round(float(order_totals.get(oid, 0.0)), 2)

    if status == "pending":
        pay_status = "pending"
    elif status == "cancelled":
        pay_status = random.choices(["failed", "refunded"], weights=[0.7, 0.3], k=1)[0]
    elif status == "returned":
        pay_status = "refunded"
    else:
        pay_status = "completed"

    pay_dt = order["order_date"] + datetime.timedelta(hours=random.randint(0, 6), minutes=random.randint(0, 59))
    payments_rows.append({
        "payment_id": pmid,
        "order_id": int(oid),
        "payment_method": random.choices(PAYMENT_METHODS, weights=PM_WEIGHTS, k=1)[0],
        "payment_status": pay_status,
        "amount": amount,
        "payment_date": pay_dt,
    })
    pmid += 1

payments_df = pd.DataFrame(payments_rows)
assert payments_df["order_id"].isin(orders_df["order_id"]).all()
payments_df.head()

,payment_id,order_id,payment_method,payment_status,amount,payment_date
0,1,1,credit_card,completed,3044.59,2026-06-04 21:04:35
1,2,2,other,completed,2003.21,2026-03-01 23:20:49
2,3,3,paypal,completed,3164.82,2024-11-09 11:01:47
3,4,4,credit_card,completed,1112.54,2025-12-20 22:10:28
4,5,5,credit_card,completed,2387.82,2026-04-21 04:35:44


## 7. `reviews` (~35% de las líneas de pedidos entregados)

In [10]:
delivered_oids = set(orders_df.loc[orders_df.order_status == "delivered", "order_id"])
delivered_items = order_items_df[order_items_df.order_id.isin(delivered_oids)]
n_reviews = int(len(delivered_items) * 0.35)
reviewed_items = delivered_items.sample(n=n_reviews, random_state=42)

delivered_date_by_order = orders_df.set_index("order_id")["delivered_date"]
fake_es = fakers_by_locale["es_ES"]

reviews_rows, rid = [], 1
for _, item in reviewed_items.iterrows():
    rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.08, 0.17, 0.35, 0.35], k=1)[0]
    has_comment = random.random() < 0.6
    review_dt = delivered_date_by_order.loc[item["order_id"]] + datetime.timedelta(days=random.randint(1, 20))
    reviews_rows.append({
        "review_id": rid,
        "order_item_id": int(item["order_item_id"]),
        "rating": rating,
        "comment": fake_es.sentence(nb_words=10) if has_comment else None,
        "review_date": review_dt,
    })
    rid += 1

reviews_df = pd.DataFrame(reviews_rows)
assert reviews_df["order_item_id"].isin(order_items_df["order_item_id"]).all()
print(f"reviews: {len(reviews_df)} filas ({100*len(reviews_df)/len(delivered_items):.1f}% de líneas entregadas)")
reviews_df.head()

reviews: 909 filas (35.0% de líneas entregadas)


,review_id,order_item_id,rating,comment,review_date
0,1,2850,4,Edad sistemas atrás costa conciencia asimismo ...,2025-10-15 18:12:25
1,2,370,5,Pocos esto qué policía media sectores idea pri...,2025-11-08 09:51:47
2,3,485,4,Puntos riesgo año asimismo dónde amigo semana ...,2024-04-27 11:26:29
3,4,3774,3,None,2026-05-05 02:28:57
4,5,2844,4,None,2025-12-30 13:25:45


## Carga en BigQuery

Función reutilizable con manejo de errores y validación de filas cargadas. Reutiliza el esquema ya definido en `01_setup_bigquery.ipynb` 

In [16]:
import decimal

def _cast_numeric_columns(df, schema):
    """Convierte a decimal.Decimal las columnas NUMERIC/BIGNUMERIC del schema.
    Necesario porque pyarrow no sabe convertir float64 -> decimal128 directamente
    (ver ArrowInvalid: Got bytestring of length 8, expected 16)."""
    df = df.copy()
    for field in schema:
        if field.field_type in ("NUMERIC", "BIGNUMERIC") and field.name in df.columns:
            df[field.name] = df[field.name].apply(
                lambda x: decimal.Decimal(str(x)) if pd.notna(x) else None
            )
    return df


def load_dataframe_to_bq(client, df, dataset_id, table_name):
    """Carga un DataFrame en una tabla de BigQuery ya existente (WRITE_TRUNCATE) y valida el resultado."""
    table_ref = f"{client.project}.{dataset_id}.{table_name}"
    try:
        existing_schema = client.get_table(table_ref).schema
    except Exception as e:
        print(f"✗ La tabla {table_name} no existe todavía. Ejecuta antes 01_setup_bigquery.ipynb. ({e})")
        raise

    df_to_load = _cast_numeric_columns(df, existing_schema)

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        schema=existing_schema,
    )

    try:
        job = client.load_table_from_dataframe(df_to_load, table_ref, job_config=job_config)
        job.result()
    except Exception as e:
        print(f"✗ Error cargando {table_name}: {e}")
        raise

    table = client.get_table(table_ref)
    expected, actual = len(df_to_load), table.num_rows
    status = "✔" if expected == actual else "⚠"
    print(f"{status} {table_name:<15} {actual} filas cargadas (esperadas: {expected})")
    return table

In [17]:
LOAD_ORDER = ["categories", "customers", "products", "orders", "order_items", "payments", "reviews"]
dataframes = {
    "categories": categories_df,
    "customers": customers_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "reviews": reviews_df,
}

for table_name in LOAD_ORDER:
    load_dataframe_to_bq(client, dataframes[table_name], DATASET_ID, table_name)


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔ categories      8 filas cargadas (esperadas: 8)


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔ customers       500 filas cargadas (esperadas: 500)


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


...

✔ order_items     4586 filas cargadas (esperadas: 4586)


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔ payments        2000 filas cargadas (esperadas: 2000)


c:\Users\GAMER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✔ reviews         909 filas cargadas (esperadas: 909)
